In [ ]:

!nvidia-smi
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# ==============================================================================
# CELL 1: Setup Environment and Mount Google Drive
# ==============================================================================
!pip install -q ultralytics

from google.colab import drive
import os
from ultralytics import YOLO

# Mount Google Drive to save our weights permanently
drive.mount('/content/drive')

# Define permanent paths in your Drive
PROJECT_DIR = '/content/drive/MyDrive/Smart_Scan/Detection_Model'
DATASET_DIR = os.path.join(PROJECT_DIR, 'dataset')  # Singular: dataset folder
RUNS_DIR = os.path.join(PROJECT_DIR, 'yolo_runs')

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

print(f"✅ Environment ready. Saving weights to: {RUNS_DIR}")


In [ ]:
# ==============================================================================
# CELL 2: Setup Dataset Directory (Extract .tgz and Create data.yaml)
# ==============================================================================
EXTRACT_PATH = os.path.join(DATASET_DIR, 'IBEM_data')
os.makedirs(EXTRACT_PATH, exist_ok=True)

# Path to your uploaded .tgz file
TGZ_PATH = os.path.join(DATASET_DIR, 'IBEM_dataset.tgz')

# Check if dataset is already extracted
data_yaml_path = os.path.join(EXTRACT_PATH, 'data.yaml')

if not os.path.exists(data_yaml_path):
    if os.path.exists(TGZ_PATH):
        print("📦 Extracting IBEM dataset from Google Drive...")
        import tarfile
        with tarfile.open(TGZ_PATH, 'r:gz') as tar:
            tar.extractall(path=EXTRACT_PATH)
        print("✅ Extraction complete!")
    else:
        print(f"❌ ERROR: IBEM_dataset.tgz not found at {TGZ_PATH}")
        print("Please upload IBEM_dataset.tgz to your Google Drive")
else:
    print(f"✅ Dataset already extracted. Found: {data_yaml_path}")

# Define DATA_YAML path for training
DATA_YAML = data_yaml_path
print(f"📍 Data YAML path: {DATA_YAML}")


## ⏱️ Training Times & GPU Pricing (YOLOv8 Detection)

### Training Time: **45-60 minutes** (50 epochs)

### Cost Breakdown per GPU:
| GPU | Price/Hour | Training Cost | Recommendation |
|-----|-----------|--------------|-----------------|
| **A100 (40GB)** | $0.54 | **$0.41-$0.54** ✅ | **BEST VALUE** |
| **A100 (80GB)** | $0.75 | $0.56-$0.75 | More VRAM, slower |
| **G4 96GB** | $0.87 | $0.65-$0.87 | Slowest, not worth it |

### 💰 With $10 Budget:
- **A100 (40GB)**: 18+ full trainings ✅
- **A100 (80GB)**: 13+ full trainings
- **G4 96GB**: 11+ full trainings

**🎯 Recommendation: Use A100 (40GB)** - Cheapest & Fastest for detection tasks!


In [ ]:
# ==============================================================================
# CELL 3: The Auto-Resume Training Loop (Colab Pro Resilient)
# ==============================================================================
# This cell handles everything. If Colab disconnects, just run this cell again.

# Path to where YOLO saves the most recent backup
LAST_CHECKPOINT = os.path.join(RUNS_DIR, 'math_detector', 'weights', 'last.pt')

# Verify data.yaml exists before training
if not os.path.exists(DATA_YAML):
    print(f"❌ ERROR: data.yaml not found at {DATA_YAML}")
    print("Run Cell 2 again to extract the dataset properly.")
else:
    if os.path.exists(LAST_CHECKPOINT):
        print(f"🔄 Previous training session found! Resuming from: {LAST_CHECKPOINT}")
        # Load the partially trained model
        model = YOLO(LAST_CHECKPOINT)
        
        # Resume training with Colab Pro settings
        results = model.train(
            resume=True,
            data=DATA_YAML,
            epochs=50,
            imgsz=640,
            batch=16,
            project=RUNS_DIR,
            name='math_detector',
            save=True,
            save_period=5,
            device=0
        )
        print("✅ Training resumed successfully!")

    else:
        print("🚀 No previous session found. Starting fresh training...")
        # Load a brand new Nano model (lightweight for Raspberry Pi deployment later)
        model = YOLO('yolov8n.pt')
        
        results = model.train(
            data=DATA_YAML,
            epochs=50,          # Total epochs to train
            imgsz=640,          # Image size
            batch=16,           # Lower this to 8 if you get "CUDA Out of Memory" errors
            project=RUNS_DIR,   # Force saving to Google Drive
            name='math_detector',
            save=True,          # Save checkpoints
            save_period=5,      # Save a backup to Drive every 5 epochs
            device=0            # Force GPU usage
        )
        print("🎉 Detection Model Training Complete!")


In [ ]:
# ==============================================================================
# CELL 4: Verify Training Completion and Model Weights
# ==============================================================================
import os

print("🔍 Verifying training results...")

# Check if model weights directory exists
weights_dir = os.path.join(RUNS_DIR, 'math_detector', 'weights')
if os.path.exists(weights_dir):
    weights_files = os.listdir(weights_dir)
    print(f"\n✅ Found trained weights in: {weights_dir}")
    print(f"   Files: {weights_files}")
    
    # Check best and last models
    has_best = 'best.pt' in weights_files
    has_last = 'last.pt' in weights_files
    print(f"\n   best.pt exists: {'✅' if has_best else '❌'}")
    print(f"   last.pt exists: {'✅' if has_last else '❌'}")
    
    if has_best:
        print("\n🎉 Training Complete! Best model ready for deployment.")
else:
    print(f"⚠️ Weights directory not found. Training may not have completed.")

print(f"\n📁 Full runs directory structure:")
for root, dirs, files in os.walk(RUNS_DIR):
    level = root.replace(RUNS_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:3]:  # Show first 3 files per directory
        print(f'{subindent}{file}')
    if len(files) > 3:
        print(f'{subindent}... and {len(files) - 3} more files')
